# Step 7 — American Options

American options can be exercised at any time before expiry. This creates a **free-boundary problem**: at each point in time, we must decide whether it is optimal to exercise early or hold.

The key modification to the PDE solver is simple: after each backward time step, enforce

$$V(S, t) = \max(V(S, t),\; (K - S)^+)$$

This is the early exercise constraint. We use Crank-Nicolson as the base scheme.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.solvers.american import solve_american, binomial_american
from src.solvers.implicit import solve_implicit
from src.core import bs_price

## Parameters

In [ ]:
K     = 100
r     = 0.05
sigma = 0.20
T     = 1.0
S0    = 100.0
S_max = 3 * K
M, N  = 300, 300

eur_put = bs_price(S0, K, T, r, sigma, option_type='put')
print(f'European put (Black-Scholes): {eur_put:.4f}')

## 1. Price the American Put

In [ ]:
S_am, V_am = solve_american(S_max, K, r, sigma, T, M, N, option_type='put')

# interpolate at S0
amer_put_fdm = float(np.interp(S0, S_am, V_am))
print(f'American put (FDM):           {amer_put_fdm:.4f}')
print(f'European put (Black-Scholes): {eur_put:.4f}')
print(f'Premium (American - European): {amer_put_fdm - eur_put:.4f}')
assert amer_put_fdm >= eur_put, 'American put must be >= European put'

## 2. Validate Against Binomial Tree

In [ ]:
bin_price = binomial_american(S0, K, r, sigma, T, N=1000, option_type='put')
print(f'American put (binomial, N=1000): {bin_price:.4f}')
print(f'American put (FDM, M=N=300):     {amer_put_fdm:.4f}')
print(f'Difference:                      {abs(amer_put_fdm - bin_price):.4f}')

## 3. Price Profile: American vs European Put

In [ ]:
_, V_eur = solve_implicit(S_max, K, r, sigma, T, M, N, option_type='put')

S_plot = S_am[S_am <= 2 * K]
idx    = len(S_plot)

plt.figure(figsize=(8, 4))
plt.plot(S_plot, V_am[:idx],          label='American put (FDM)')
plt.plot(S_plot, V_eur[:idx], '--',   label='European put (FDM)')
plt.plot(S_plot, np.maximum(K - S_plot, 0), ':', color='gray', label='Intrinsic value')
plt.axvline(K, color='k', lw=0.8, ls='--', alpha=0.4)
plt.xlabel('Stock price S')
plt.ylabel('Option value')
plt.title('American vs European Put')
plt.legend()
plt.grid(True, ls='--', alpha=0.4)
plt.tight_layout()
plt.savefig('../results/07_american_vs_european.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Early Exercise Premium vs Stock Price

The early exercise premium is largest for deep in-the-money puts, where immediate exercise becomes optimal.

In [ ]:
premium = V_am[:idx] - V_eur[:idx]

plt.figure(figsize=(8, 3))
plt.plot(S_plot, premium)
plt.axvline(K, color='k', lw=0.8, ls='--', alpha=0.4, label='K=100')
plt.xlabel('Stock price S')
plt.ylabel('Early exercise premium')
plt.title('Early Exercise Premium (American - European)')
plt.legend()
plt.grid(True, ls='--', alpha=0.4)
plt.tight_layout()
plt.savefig('../results/07_early_exercise_premium.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Convergence vs Binomial Tree

In [ ]:
ref = binomial_american(S0, K, r, sigma, T, N=5000, option_type='put')
print(f'Reference (binomial N=5000): {ref:.6f}')

grid_sizes = [30, 60, 90, 120, 150, 180, 210, 300]
errors = []
for M in grid_sizes:
    S, V = solve_american(S_max, K, r, sigma, T, M, M, option_type='put')
    price = float(np.interp(S0, S, V))
    errors.append(abs(price - ref))
    print(f'M=N={M:3d}  price={price:.6f}  error={errors[-1]:.6f}')

plt.figure(figsize=(7, 4))
plt.loglog(grid_sizes, errors, 'o-')
plt.xlabel('Grid size M (= N)')
plt.ylabel('Absolute error vs binomial reference')
plt.title('American Put Convergence')
plt.grid(True, which='both', ls='--', alpha=0.5)
plt.tight_layout()
plt.savefig('../results/07_convergence.png', dpi=150, bbox_inches='tight')
plt.show()